# EDA Tools Agent

A tool-calling agent for exploratory data analysis, modelled on
[business-science/ai-data-science-team's EDA Tools Agent](https://github.com/business-science/ai-data-science-team/blob/master/examples/ds_agents/eda_tools_agent.ipynb).

The agent picks from these tools:

| Reference tools | Extended tools |
|---|---|
| `explain_data`, `describe_dataset` | `assess_data_quality`: 0-100 score, issues, suggested fixes |
| `visualize_missing` (missingno) | `detect_outliers`: IQR / z-score |
| `generate_correlation_funnel` | `analyze_correlations`: matrix + multicollinearity |
| `generate_sweetviz_report`, `generate_dtale_report` | `profile_distributions`: skew, normality, transforms |
| | `analyze_relationship`: automatic statistical test between two columns |

The dataset is never sent to the LLM. Tools receive it from agent state, and only their text summaries enter the conversation.

## Load libraries and data

In [ ]:
import os
import base64
from io import BytesIO

import pandas as pd
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

from eda_agents.agents.eda_tools import EDAToolsAgent
from eda_agents.utils.plotting import plotly_from_dict
from eda_agents.utils.sample_data import make_sample_churn_data

df = make_sample_churn_data()   # or: pd.read_csv("your_data.csv")
df.head()

## Setup AI
Any LangChain tool-calling chat model works (OpenAI, Anthropic, Ollama, ...).

In [ ]:
# os.environ["OPENAI_API_KEY"] = "..."
llm = ChatOpenAI(model="gpt-4o-mini")

## Create the agent

In [ ]:
eda_agent = EDAToolsAgent(llm, invoke_react_agent_kwargs={"recursion_limit": 25})
eda_agent.show(xray=1)

## Usage
#### What tools do you have access to?

In [ ]:
eda_agent.invoke_agent("What tools do you have access to? Return a table.", data_raw=df)
eda_agent.get_ai_message(markdown=True)

## Tool examples
#### Example 1: Describe the dataset

In [ ]:
eda_agent.invoke_agent("Describe the dataset with the describe_dataset tool.", data_raw=df)
eda_agent.get_artifacts(as_dataframe=True)["describe_dataset"]["describe_df"]

#### Example 2: Missing data

In [ ]:
eda_agent.invoke_agent("Visualize missing data in the dataset.", data_raw=df)
print(eda_agent.get_ai_message())
display(Image(base64.b64decode(eda_agent.get_artifacts()["visualize_missing"]["matrix_plot"])))

#### Example 3: Correlation funnel
Uses pytimetk when installed, otherwise a built-in pandas implementation.

In [ ]:
eda_agent.invoke_agent(
    "Use the correlation funnel tool with Churn as the target and Churn=Yes as the bin to compare against.",
    data_raw=df,
)
display(eda_agent.get_ai_message(markdown=True))
plotly_from_dict(eda_agent.get_artifacts()["generate_correlation_funnel"]["plotly_figure"])

#### Example 4: Data quality score (new)

In [ ]:
eda_agent.invoke_agent("Assess the data quality and tell me what to fix first.", data_raw=df)
display(eda_agent.get_ai_message(markdown=True))
quality = eda_agent.get_artifacts()["assess_data_quality"]
print(f"Score: {quality['score']} ({quality['grade']})")
pd.DataFrame(quality["issues"])

#### Example 5: Statistical tests (new)
The agent can call a tool several times. Repeated artifacts are keyed `name`, `name_2`, ...

In [ ]:
eda_agent.invoke_agent("Does contract type affect churn? Is tenure different for churners?", data_raw=df)
display(eda_agent.get_ai_message(markdown=True))
print(eda_agent.get_tool_calls())
for name, result in eda_agent.get_artifacts().items():
    print(name, "->", result["test"], "|", result["conclusion"])
plotly_from_dict(eda_agent.get_artifacts()["analyze_relationship"]["plotly_figure"])

#### Example 6: Outliers and distributions (new)

In [ ]:
eda_agent.invoke_agent("Profile the distributions and find outliers.", data_raw=df)
display(eda_agent.get_ai_message(markdown=True))
display(eda_agent.get_artifacts(as_dataframe=True)["profile_distributions"]["summary"])
plotly_from_dict(eda_agent.get_artifacts()["detect_outliers"]["plotly_figure"])

#### Example 7: Sweetviz report

In [ ]:
eda_agent.invoke_agent("Generate a Sweetviz report for the dataset with Churn as the target.", data_raw=df)
eda_agent.get_artifacts()["generate_sweetviz_report"]["report_file"]

#### Inspect the full trace

In [ ]:
eda_agent.get_internal_messages(markdown=True)